# Comparación inicial: Random Forest y red de atención

**Objetivo:** RF y MLP con atención; 9 variables, partición aleatoria.

Este cuaderno contiene el código completo del experimento `train_comparative_models.py`,
dividido en etapas y acompañado de explicaciones. No necesita ZIP, clonar un
repositorio ni ejecutar un script oculto. Ejecuta las celdas en orden.

**Recorrido:** entorno → carga e inspección → preparación → modelo o análisis →
evaluación → gráficos → descarga y colaboración.

**Entradas:** `dataset_entrenamiento_completo.csv`.

Los datos no se incluyen. Se conservan los valores predeterminados del protocolo histórico.
Las salidas se generarán al ejecutar; las figuras históricas no están incrustadas.


**Interpretación:** los pesos de atención, la importancia Gini y SHAP son indicadores distintos. No equivalen a causas físicas. El módulo de atención se conserva para documentar el experimento; su revisión debe evaluarse como una variante nueva.

## 1. Entorno y parámetros
Modifica los parámetros aquí para que una colaboración sea fácil de revisar. Los valores originales siguen seleccionados. Cada ejecución crea una carpeta nueva.

In [ ]:
# Instala únicamente las bibliotecas que falten en el entorno.
import importlib.util
import importlib.metadata
import subprocess
import sys
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image

DEPENDENCIAS = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'torch': 'torch'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *faltantes], check=True)

CONFIG = dict(seed=42, rf_trees=100, rf_depth=15, epochs=50,
              learning_rate=0.002, batch_size=256)
EXPERIMENTO = 'comparacion_inicial'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = Path(globals().get('DATA_DIR_OVERRIDE', 'data')).resolve()
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR_OVERRIDE', Path('results') / EXPERIMENTO / RUN_ID)).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
print('Parámetros:', CONFIG)
print('Salidas:', OUTPUT_DIR)

def mostrar_figura(ruta):
    # Conserva la figura en disco y también la muestra en el cuaderno.
    if str(ruta).lower().endswith('.png'):
        display(Image(filename=str(ruta)))


## 2. Carga e inspección de los datos
En Colab, la siguiente celda ofrece el selector si faltan archivos. Carga solo los CSV indicados, conservando sus nombres. En un entorno local, colócalos en `data/`. Para el dataset realista usa la copia de la subcarpeta `resultados` original.

In [ ]:
import pandas as pd

ARCHIVOS_REQUERIDOS = ['dataset_entrenamiento_completo.csv']
def localizar(nombre):
    candidatos = [DATA_DIR / nombre, DATA_DIR / 'resultados' / nombre]
    return next((p for p in candidatos if p.is_file()), candidatos[0])

pendientes = [n for n in ARCHIVOS_REQUERIDOS if not localizar(n).is_file()]
if pendientes:
    print('Archivos requeridos:', pendientes)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloca los CSV en ' + str(DATA_DIR))
    cargados = files.upload()
    for nombre, contenido in cargados.items():
        if Path(nombre).name != nombre or nombre not in ARCHIVOS_REQUERIDOS:
            raise ValueError('Archivo inesperado: ' + nombre)
        (DATA_DIR / nombre).write_bytes(contenido)
    del cargados

DATA_FILES = {n: localizar(n) for n in ARCHIVOS_REQUERIDOS}
if any(not p.is_file() for p in DATA_FILES.values()):
    raise FileNotFoundError('Faltan CSV; repite esta celda y carga los indicados.')

# Inspección visible: dimensiones, tipos, valores faltantes y balance de clases.
for nombre, ruta in DATA_FILES.items():
    tabla = pd.read_csv(ruta)
    print('\n', nombre, '| filas y columnas:', tabla.shape)
    display(tabla.head())
    display(pd.DataFrame({'tipo': tabla.dtypes.astype(str), 'nulos': tabla.isna().sum()}))
    if 'clase' in tabla.columns:
        display(tabla['clase'].value_counts(dropna=False).rename('filas').to_frame())

REGISTRO = dict(experimento=EXPERIMENTO, parametros=CONFIG, run_id=RUN_ID,
    input_sha256={n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in DATA_FILES.items()},
    versiones={p: importlib.metadata.version(p) for p in DEPENDENCIAS.values()},
    estado='iniciado')
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
BASE_DIR = str(OUTPUT_DIR)
FILE_COMBINADO = str(DATA_FILES['dataset_entrenamiento_completo.csv'])
FILE_CSV_COMP = str(OUTPUT_DIR / 'comparativa_factores.csv')
FILE_PNG_COMP = str(OUTPUT_DIR / 'comparativa_modelos_factores.png')
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])


### Definición: `FeatureSelfAttention`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FeatureSelfAttention(nn.Module):
    def __init__(self, input_dim):
        super(FeatureSelfAttention, self).__init__()
        self.query = nn.Linear(input_dim, input_dim, bias=False)
        self.key = nn.Linear(input_dim, input_dim, bias=False)
        self.value = nn.Linear(input_dim, input_dim, bias=False)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        # x: [batch_size, input_dim]
        # Proyectar Q, K, V
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        
        # Calcular scores de atención (similitud Q y K)
        # Scaled dot-product attention simplificado para variables tabulares
        attention_scores = torch.matmul(q.unsqueeze(2), k.unsqueeze(1)) / np.sqrt(x.shape[1])
        attention_weights = self.softmax(attention_scores.mean(dim=2)) # Promediar pesos
        
        # Aplicar pesos al valor
        attended_x = v * attention_weights
        return attended_x, attention_weights


### Definición: `FireAttentionMLP`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FireAttentionMLP(nn.Module):
    def __init__(self, input_dim):
        super(FireAttentionMLP, self).__init__()
        self.attention = FeatureSelfAttention(input_dim)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attended_x, weights = self.attention(x)
        output = self.classifier(attended_x)
        return output, weights


## 4. Desarrollo del experimento
Las siguientes celdas contienen el cuerpo del análisis original, con comentarios y código editable. Puedes inspeccionar las variables entre pasos. Ejecuta nuevamente desde la carga de datos si cambias una transformación previa.

### Etapa 1: Partición y preparación de muestras
La división aleatoria usa una semilla fija. El escalador se ajusta solo sobre entrenamiento. Una partición aleatoria no demuestra transferencia geográfica.

In [ ]:
print("="*70)
print("  ANÁLISIS COMPARATIVO PARA PAPER: MACHINE LEARNING VS DEEP LEARNING")
print("="*70)

if not os.path.exists(FILE_COMBINADO):
    print(f"[ERROR] No se encuentra el archivo: {FILE_COMBINADO}")
    raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
# 1. Cargar y preparar datos
df = pd.read_csv(FILE_COMBINADO)
features = ['elevacion', 'pendiente', 'orientacion', 'cobertura_suelo', 'ndvi', 'evi', 'nbr', 'temp_c', 'precip_mm']

nombres_es = {
    'precip_mm': 'Precipitación',
    'temp_c': 'Temperatura',
    'nbr': 'NBR (Quemado)',
    'elevacion': 'Elevación',
    'cobertura_suelo': 'Cobertura de Suelo',
    'ndvi': 'NDVI (Verdor)',
    'evi': 'EVI (Biomasa)',
    'pendiente': 'Pendiente',
    'orientacion': 'Orientación'
}

X = df[features].values
y = df['clase'].values

# Dividir en entrenamiento (80%) y test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG['seed'], stratify=y)


### Etapa 2: Ajuste del modelo
Aquí se aprende a partir de las muestras de entrenamiento. Revisa los hiperparámetros visibles y evita ajustar decisiones usando repetidamente los resultados del conjunto de prueba.

In [ ]:

# Escalar datos (Esencial para redes neuronales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------------------------------
# MODELO 1: RANDOM FOREST
# -------------------------------------------------------------------------
print("\n[1/3] Entrenando Random Forest Classifier...")
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=CONFIG['rf_trees'], max_depth=CONFIG['rf_depth'], random_state=CONFIG['seed'], n_jobs=-1)
rf.fit(X_train, y_train) # RF entrena con datos sin escalar (preferible para árboles)

# Evaluar RF
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

metrics_rf = {
    'Accuracy': accuracy_score(y_test, rf_pred),
    'Precision': precision_score(y_test, rf_pred),
    'Recall': recall_score(y_test, rf_pred),
    'F1-Score': f1_score(y_test, rf_pred),
    'ROC-AUC': roc_auc_score(y_test, rf_proba)
}
rf_importances = rf.feature_importances_


### Etapa 3: Entrenamiento de la red
Se optimizan los pesos con los datos de entrenamiento. El registro muestra la pérdida de la última tanda; no es una métrica de validación. Mantén fijo el conjunto de prueba al comparar configuraciones.

In [ ]:

# -------------------------------------------------------------------------
# MODELO 2: RED NEURONAL CON ATENCIÓN (PyTorch)
# -------------------------------------------------------------------------
print("\n[2/3] Entrenando Red Neuronal con Mecanismo de Atención (PyTorch)...")

# Crear datasets de PyTorch
train_dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train).unsqueeze(1))
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)

X_test_t = torch.FloatTensor(X_test_scaled)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

# Inicializar red, optimizador y pérdida
model = FireAttentionMLP(input_dim=len(features))
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-5)

# Bucle de entrenamiento (50 epochs)
model.train()
for epoca in range(CONFIG['epochs']):
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs, _ = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if epoca == 0 or (epoca + 1) % 10 == 0 or epoca + 1 == CONFIG['epochs']:
        print(f"Época {epoca + 1}/{CONFIG['epochs']} | pérdida última tanda: {float(loss.detach()):.5f}")


### Etapa 4: Evaluación
Estas métricas pertenecen al protocolo y a la partición de este cuaderno. Revisa precisión, recall y matriz de confusión además del accuracy.

In [ ]:

# Evaluar Red Neuronal
model.eval()
with torch.no_grad():
    nn_outputs, nn_weights = model(X_test_t)
    nn_pred_proba = nn_outputs.numpy()
    nn_pred = (nn_pred_proba >= 0.5).astype(int)

    # Calcular importancia promedio a partir de los pesos del mecanismo de atención
    nn_attention_weights = nn_weights.mean(dim=0).numpy()
    # Normalizar pesos de atención para que sumen 100% (para comparación justa)
    nn_importances = nn_attention_weights / np.sum(nn_attention_weights)

metrics_nn = {
    'Accuracy': accuracy_score(y_test, nn_pred),
    'Precision': precision_score(y_test, nn_pred),
    'Recall': recall_score(y_test, nn_pred),
    'F1-Score': f1_score(y_test, nn_pred),
    'ROC-AUC': roc_auc_score(y_test, nn_pred_proba)
}

# -------------------------------------------------------------------------
# COMPARAR MÉTRICAS DE RENDIMIENTO
# -------------------------------------------------------------------------
print("\n" + "="*50)
print("  TABLA COMPARATIVA DE MÉTRICAS EN TEST SET")


### Etapa 5: Inspección y cálculo intermedio
Ejecuta esta etapa después de la anterior y revisa los valores impresos. Las variables quedan disponibles para explorarlas o modificarlas en otra celda.

In [ ]:
print("="*50)
df_metrics = pd.DataFrame([metrics_rf, metrics_nn], index=['Random Forest', 'Attention MLP'])
print(df_metrics.round(5))

# -------------------------------------------------------------------------
# COMPARAR FACTORES DE IMPORTANCIA
# -------------------------------------------------------------------------
print("\n" + "="*50)
print("  COMPARATIVA DE IMPORTANCIA DE VARIABLES (%)")
print("="*50)
df_comp = pd.DataFrame({
    'Variable': [nombres_es[f] for f in features],
    'Random Forest (Gini)': rf_importances * 100,
    'Attention Net (Pesos)': nn_importances * 100
})
# Ordenar por importancia de Random Forest
df_comp = df_comp.sort_values(by='Random Forest (Gini)', ascending=False).reset_index(drop=True)
print(df_comp.round(2))

# Guardar en CSV para tablas del paper
df_comp.to_csv(FILE_CSV_COMP, index=False)
print(f"\n[OK] CSV comparativo guardado en: {FILE_CSV_COMP}")

# -------------------------------------------------------------------------
# GRAFICAR COMPARATIVA PARA EL PAPER
# -------------------------------------------------------------------------
print("\n[3/3] Generando gráfico comparativo...")


### Etapa 6: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:

plt.figure(figsize=(12, 7), dpi=300)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Configuración de las barras dobles
y_pos = np.arange(len(features))
width = 0.35 # Ancho de barra

# Ordenar los datos para graficar de forma ascendente
df_plot = df_comp.sort_values(by='Random Forest (Gini)', ascending=True).reset_index(drop=True)

rects1 = plt.barh(y_pos - width/2, df_plot['Random Forest (Gini)'], width, label='Random Forest (Global)', color='#d95f0e', edgecolor='none')
rects2 = plt.barh(y_pos + width/2, df_plot['Attention Net (Pesos)'], width, label='Red de Atención (Local Prom.)', color='#2b8cbe', edgecolor='none')

plt.yticks(y_pos, df_plot['Variable'], fontsize=11, fontweight='semibold')
plt.xticks(fontsize=10)
plt.xlabel('Importancia / Peso del Factor (%)', fontsize=12, fontweight='bold', labelpad=10)
plt.title('Comparación de Factores de Incendio: Random Forest vs. Mecanismo de Autotensión', 
          fontsize=14, fontweight='bold', pad=15, color='#111111')
plt.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='#e0e0e0', fontsize=11)

# Quitar bordes para diseño limpio
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)


### Etapa 7: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:
ax.spines['bottom'].set_color('#cccccc')
ax.yaxis.grid(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.5, color='#cccccc')

plt.tight_layout()
plt.savefig(FILE_PNG_COMP, bbox_inches='tight')
mostrar_figura(FILE_PNG_COMP)
plt.close()

print(f"[OK] Gráfico comparativo guardado en: {FILE_PNG_COMP}")
print("="*70)


## 5. Inspeccionar y guardar resultados
Las tablas se muestran debajo. El registro identifica parámetros, versiones y hashes. Esta celda marca el registro como completado: ejecútala solo cuando hayan terminado todas las etapas anteriores.

In [ ]:
for ruta in sorted(OUTPUT_DIR.glob('*.csv')):
    print(ruta.name)
    display(pd.read_csv(ruta).head(20))
REGISTRO['estado'] = 'completado'
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salidas:', [p.name for p in sorted(OUTPUT_DIR.iterdir())])


In [ ]:
# Descarga las salidas antes de cerrar una sesión de Colab.
import shutil
paquete = shutil.make_archive(str(OUTPUT_DIR.parent / (EXPERIMENTO + '-' + RUN_ID)), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
except ImportError:
    print('ZIP guardado en:', paquete)
else:
    files.download(paquete)


## 6. Proponer una colaboración

1. Plantea una pregunta concreta: por ejemplo, el efecto de un hiperparámetro o de una variable.
2. Conserva una ejecución con la configuración original y luego cambia una sola decisión.
3. Registra el hash del dataset, la partición, la semilla y todas las métricas relevantes.
4. En GitHub, abre un Issue para discutir la propuesta. Trabaja en un fork o en una rama
   con permisos y envía un pull request que explique el cambio y su evidencia.
5. Antes del commit, borra las salidas del cuaderno y revisa que no incluyan datos privados.
   Adjunta resultados únicamente cuando tengas permiso para compartirlos.

Edita la siguiente celda Markdown para documentar tu propuesta. No se envía nada automáticamente.

### Registro de la propuesta

- **Autor y objetivo:**
- **Hipótesis:**
- **Parámetro o código modificado:**
- **Dataset y hash:**
- **Semilla y partición:**
- **Resultado original y resultado de la variante:**
- **Limitaciones y conclusión:**
